In [2]:
import sqlite3, shutil
from datetime import date
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()
DB_PATH = (BASE_DIR / "cleaning.db").resolve()
CSV_PATH = BASE_DIR / "cleaning_usaha_flag3_16_oktober.csv"  # sesuaikan lokasi CSV

TABEL = "usaha"
KEYS = ["assignment_id", "nama_usaha"]
STATUS_COL = "status_penyelesaian"
KET_COL = "keterangan"
UPDATED_COL = "update_at"
FLAG_COL = "flag_3_new"
L4_COL = "level_4_full_code"
L2_COL = "level_2_code"

KET_SELESAI = "sudah selesai akibat agregat kbli sudah dalam range rasio flag 3"

# Baris DB yang TIDAK ada di CSV baru ditandai selesai, dengan batasan:
HANYA_FLAG3 = True  # hanya baris yang flag_3_new = 1
LEWATI_SUDAH_SELESAI = (
    True  # jangan timpa baris yang status-nya sudah 1 (keterangan lama aman)
)

assert DB_PATH.exists(), f"Database tidak ditemukan: {DB_PATH}"
assert CSV_PATH.exists(), f"CSV tidak ditemukan: {CSV_PATH}"
print("DB :", DB_PATH)
print("CSV:", CSV_PATH)

DB : D:\2026\Sensus Ekonomi\cleaning_data\3 Oktober 2026\Mikro\RantabDBNAS\cleaning.db
CSV: d:\2026\Sensus Ekonomi\cleaning_data\3 Oktober 2026\Mikro\RantabDBNAS\cleaning_usaha_flag3_16_oktober.csv


In [3]:
# Backup dulu (sekali per tanggal, kalau belum ada)
tanggal = date.today().isoformat()
backup = DB_PATH.with_name(f"{DB_PATH.stem}_backup_{tanggal}.db")

if not backup.exists():
    shutil.copy2(DB_PATH, backup)
    print("Backup dibuat:", backup)
else:
    print("Backup sudah ada:", backup)

Backup dibuat: D:\2026\Sensus Ekonomi\cleaning_data\3 Oktober 2026\Mikro\RantabDBNAS\cleaning_backup_2026-10-16.db


In [ ]:
# RESTORE - jalankan HANYA kalau mau mengembalikan DB dari backup
try:
    conn.close()
except Exception:
    pass

shutil.copy2(backup, DB_PATH)
print("DB dikembalikan dari:", backup)

In [4]:
# Buka DB + tambah kolom baru
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

info = cur.execute(f'PRAGMA table_info("{TABEL}")').fetchall()
db_cols = [r[1] for r in info]

for kol in (L4_COL, UPDATED_COL):
    if kol not in db_cols:
        cur.execute(f'ALTER TABLE "{TABEL}" ADD COLUMN "{kol}" TEXT')
        db_cols.append(kol)
        print("Kolom ditambahkan:", kol)
conn.commit()

for kol in KEYS + [STATUS_COL, KET_COL, L2_COL, FLAG_COL]:
    assert kol in db_cols, f"Kolom '{kol}' tidak ada di tabel {TABEL}"

print("Total baris DB:", cur.execute(f'SELECT COUNT(*) FROM "{TABEL}"').fetchone()[0])

Kolom ditambahkan: level_4_full_code
Total baris DB: 1925018


In [5]:
df = pd.read_csv(CSV_PATH, dtype=str)

for k in KEYS + [L4_COL]:
    assert k in df.columns, f"Kolom '{k}' tidak ada di CSV"

# level_2_code = 4 digit pertama level_4_full_code (CSV tidak punya level_2)
kode = (
    df[L4_COL].fillna("").astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
)
df[L4_COL] = kode.where(kode != "", None)
df[L2_COL] = kode.str[:4].where(kode != "", None)

print(df.shape)
print(f"{L4_COL} kosong di CSV:", df[L4_COL].isna().sum())
df[KEYS + [L4_COL, L2_COL]].head()

(324166, 25)
level_4_full_code kosong di CSV: 0


,assignment_id,nama_usaha,level_4_full_code,level_2_code
0,000009a7-fe00-41e5-bee2-0cc07cc65dc4,USAHA AIR REBUSAN <NURHIDAYAH>,1602022006,1602
1,00000ee5-68a2-4eb2-ac93-2234dc24489d,WARUNG SEMBAKO (RONAL),1608010001,1608
2,00004c39-e36d-4e08-8acc-4270d009ce67,WARUNG HODIJAH,1671020013,1671
3,00010432-0be4-40b5-be6a-ebc6c05070e8,Warung manisan (jajang suparman),1603033006,1603
4,00012661-ab08-436b-af8a-75537385a0a9,Konter pulsa,1673020007,1673


In [6]:
# Index supaya pencarian per baris tidak full-scan
key_sql = ", ".join(f'"{k}"' for k in KEYS)
cur.execute(f'CREATE INDEX IF NOT EXISTS idx_keys_flag3 ON "{TABEL}" ({key_sql})')
conn.commit()

where_sql = " AND ".join(f'"{k}" IS ?' for k in KEYS)

cocok = set()  # rowid DB yang ketemu di CSV
n_update = n_csv_tidak_ada = 0

try:
    # 1) Ketemu -> update level_4_full_code sesuai CSV
    for _, rec in df.iterrows():
        key_vals = [None if pd.isna(rec[k]) else str(rec[k]) for k in KEYS]
        l4 = None if pd.isna(rec[L4_COL]) else rec[L4_COL]
        l2 = None if pd.isna(rec[L2_COL]) else rec[L2_COL]

        rids = [
            r[0]
            for r in cur.execute(
                f'SELECT rowid FROM "{TABEL}" WHERE {where_sql}', key_vals
            )
        ]

        if not rids:
            n_csv_tidak_ada += 1
            continue

        for rid in rids:
            cur.execute(
                f'UPDATE "{TABEL}" SET "{L4_COL}" = ?, '
                f'"{L2_COL}" = COALESCE(NULLIF(TRIM("{L2_COL}"), \'\'), ?) '
                f"WHERE rowid = ?",
                (l4, l2, rid),
            )
            cocok.add(rid)
            n_update += 1

    # 2) Tidak ketemu di CSV -> tandai selesai
    cur.execute("CREATE TEMP TABLE IF NOT EXISTS _cocok (rid INTEGER PRIMARY KEY)")
    cur.execute("DELETE FROM _cocok")
    cur.executemany("INSERT INTO _cocok VALUES (?)", [(r,) for r in cocok])

    syarat = ["rowid NOT IN (SELECT rid FROM _cocok)"]
    if HANYA_FLAG3:
        syarat.append(f'CAST("{FLAG_COL}" AS REAL) = 1')
    if LEWATI_SUDAH_SELESAI:
        syarat.append(f'COALESCE("{STATUS_COL}", 0) <> 1')

    sql_selesai = f"""
        UPDATE "{TABEL}"
        SET "{STATUS_COL}" = 1,
            "{KET_COL}" = ?,
            "{UPDATED_COL}" = datetime('now', '+7 hours')
        WHERE {" AND ".join(syarat)}
    """
    cur.execute(sql_selesai, (KET_SELESAI,))
    n_selesai = cur.rowcount

    cur.execute("DROP TABLE IF EXISTS _cocok")
    conn.commit()
    print(
        f"Selesai. level_4 terupdate: {n_update} baris DB | "
        f"ditandai selesai (tidak ada di CSV): {n_selesai} | "
        f"baris CSV tidak ada di DB (dilewati): {n_csv_tidak_ada}"
    )
except Exception:
    conn.rollback()
    print("Error - perubahan pada transaksi ini di-rollback")
    raise

Selesai. level_4 terupdate: 323461 baris DB | ditandai selesai (tidak ada di CSV): 23086 | baris CSV tidak ada di DB (dilewati): 708


In [7]:
# Verifikasi
print(
    f"{L4_COL} terisi :",
    cur.execute(
        f'SELECT COUNT(*) FROM "{TABEL}" WHERE "{L4_COL}" IS NOT NULL'
    ).fetchone()[0],
)
print(
    "Ditandai selesai :",
    cur.execute(
        f'SELECT COUNT(*) FROM "{TABEL}" WHERE "{KET_COL}" = ?', (KET_SELESAI,)
    ).fetchone()[0],
)
print(
    f"{L2_COL} NULL   :",
    cur.execute(f'SELECT COUNT(*) FROM "{TABEL}" WHERE {L2_COL} IS NULL').fetchone()[0],
)

pd.set_option("display.max_columns", None)
display(
    pd.read_sql(
        f'SELECT {", ".join(chr(34)+k+chr(34) for k in KEYS)}, "{L4_COL}", "{L2_COL}", '
        f'"{STATUS_COL}", "{KET_COL}", "{UPDATED_COL}" '
        f'FROM "{TABEL}" WHERE "{KET_COL}" = ? LIMIT 10',
        conn,
        params=(KET_SELESAI,),
    )
)
conn.close()

level_4_full_code terisi : 323355
Ditandai selesai : 23086
level_2_code NULL   : 0


,assignment_id,nama_usaha,level_4_full_code,level_2_code,status_penyelesaian,keterangan,update_at
0,00df7b5e-96b6-4e9c-9165-207d98c1ac6f,UTP PETERNAKAN SAPI POTONG (SARGIYONO),None,1601,1,sudah selesai akibat agregat kbli sudah dalam ...,2026-10-16 09:39:44
1,2485a1a3-6e8a-4a05-8f6a-11833dc97ade,UTP PETERNAKAN SAPI (TUMIO),None,1601,1,sudah selesai akibat agregat kbli sudah dalam ...,2026-10-16 09:39:44
2,3d9d23fb-dc77-4e3e-8db8-001920f3ec9b,UTP PETERNAKAN SAPI POTONG ( FARIT ROHMAN ),None,1601,1,sudah selesai akibat agregat kbli sudah dalam ...,2026-10-16 09:39:44
3,42920dd7-64a9-44d6-93f6-f198021444a4,UTP PETERNAKAN SAPI (YUDO WIDIARNO),None,1601,1,sudah selesai akibat agregat kbli sudah dalam ...,2026-10-16 09:39:44
4,7602417a-ab92-4ca2-8f21-a296886198f1,UTP PETERNAKAN ASNAWI,None,1601,1,sudah selesai akibat agregat kbli sudah dalam ...,2026-10-16 09:39:44
5,a369f7ba-c388-47a4-84dd-5b024891005e,UTP PERTERNAKAN HENDRI YANTO,None,1601,1,sudah selesai akibat agregat kbli sudah dalam ...,2026-10-16 09:39:44
6,bec70b2f-73ee-425a-8a98-fdb471356cf9,UTP PETERNAKAN SAPI POTONG (JUMARI),None,1601,1,sudah selesai akibat agregat kbli sudah dalam ...,2026-10-16 09:39:44
7,f1e34c07-14c5-46cc-8a44-b1ecb707ea6d,PETERNAKAN (AHMAD RIFAI),None,1601,1,sudah selesai akibat agregat kbli sudah dalam ...,2026-10-16 09:39:44
8,2f258e52-1b62-4277-962a-30f44edaf9be,UTP PETERNAKAN KAMBING (AHMAD SUSANTO),None,1601,1,sudah selesai akibat agregat kbli sudah dalam ...,2026-10-16 09:39:44
9,b531a05e-e244-4bb2-b422-32728fe42e3b,UTP PETERNAKAN KAMBING (SUNARYO),None,1601,1,sudah selesai akibat agregat kbli sudah dalam ...,2026-10-16 09:39:44
